# Backup Lab (Plan B): Learning Rate Tuning with SGD (No Transformers)

Use this notebook if Hugging Face downloads are blocked or transformer fine‑tuning is too slow on learner machines.

This still teaches the **Unit 6 optimisation story** with the correct vocabulary:

- we measure wrongness with a **loss function**
- we take downhill steps using **stochastic gradient descent (SGD)**
- the **learning rate** controls step size
- we watch the **generalisation gap** (train vs validation)

Because TF‑IDF + SGD trains fast, you can see the effect of learning rate very clearly.


In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss
import matplotlib.pyplot as plt

np.random.seed(42)


In [ ]:
# Load local synthetic dataset (offline-friendly)
# Local dataset path (robust)
csv_candidates = [
    os.path.join("lab", "data", "synth_financial_sentiment.csv"),
    os.path.join("data", "synth_financial_sentiment.csv"),
]
csv_path = next((p for p in csv_candidates if os.path.exists(p)), None)
if csv_path is None:
    raise FileNotFoundError(f"Could not find synth dataset. Tried: {csv_candidates}")

# Optional: add label noise to make learning-rate effects more visible (set to 0.0 for clean data)
NOISE_FRACTION = 0.25
df = pd.read_csv(csv_path)

# Inject label noise (optional)
if NOISE_FRACTION and NOISE_FRACTION > 0:
    rng = np.random.default_rng(42)
    flip_idx = rng.choice(df.index, size=int(NOISE_FRACTION * len(df)), replace=False)
    labels = ["negative", "neutral", "positive"]
    for idx in flip_idx:
        current = df.at[idx, "label"]
        other = [l for l in labels if l != current]
        df.at[idx, "label"] = rng.choice(other)


label_order = ["negative", "neutral", "positive"]
label2id = {name: i for i, name in enumerate(label_order)}
df["y"] = df["label"].map(label2id)

X_train, X_val, y_train, y_val = train_test_split(
    df["sentence"].to_numpy(), df["y"].to_numpy(),
    test_size=0.2, random_state=42, stratify=df["y"].to_numpy()
)

vectorizer = TfidfVectorizer(ngram_range=(1,2), min_df=2)
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)

X_train_vec.shape, X_val_vec.shape


## Tune the learning rate (eta0)

We will try:
- TOO_LOW
- JUST_RIGHT
- TOO_HIGH

We train for a few epochs using `partial_fit` so we can plot loss over time.


In [ ]:
ETA_MAP = {
    "TOO_LOW": 1e-4,
    "JUST_RIGHT": 1e-2,
    "TOO_HIGH": 5.0,
}

EPOCHS = 10


In [ ]:
def run_sgd_experiment(eta0: float):
    clf = SGDClassifier(
        loss="log_loss",
        learning_rate="constant",
        eta0=eta0,
        alpha=1e-4,
        random_state=42,
    )
    classes = np.array([0,1,2])
    train_losses=[]
    val_losses=[]
    val_acc=[]
    val_f1=[]

    # initialise
    clf.partial_fit(X_train_vec, y_train, classes=classes)

    for _ in range(EPOCHS):
        clf.partial_fit(X_train_vec, y_train)
        # predict_proba exists for log_loss
        train_prob = clf.predict_proba(X_train_vec)
        val_prob = clf.predict_proba(X_val_vec)

        train_losses.append(log_loss(y_train, train_prob, labels=classes))
        val_losses.append(log_loss(y_val, val_prob, labels=classes))

        y_pred = np.argmax(val_prob, axis=1)
        val_acc.append(accuracy_score(y_val, y_pred))
        val_f1.append(f1_score(y_val, y_pred, average="macro"))

    return train_losses, val_losses, val_acc, val_f1

results = {}
for name, eta in ETA_MAP.items():
    results[name] = run_sgd_experiment(eta)

{key: (results[key][2][-1], results[key][3][-1]) for key in results}  # (val acc, val f1)


In [ ]:
plt.figure()
for name in ["TOO_LOW","JUST_RIGHT","TOO_HIGH"]:
    train_losses, val_losses, _, _ = results[name]
    plt.plot(range(1, EPOCHS+1), val_losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.title("Validation loss vs epoch (SGD)")
plt.legend(list(results.keys()))
plt.tight_layout()
plt.show()


## Debrief prompts

- Which learning rate converged fastest?
- Which was unstable (oscillations / divergence)?
- What happened to the **generalisation gap**?
- How does this relate to Unit 6’s “step size” metaphor?
- If you had to explain this to a stakeholder, what would you say?
